Hi! I am a Kaggle beginner and not fluent in English, so this notebook is written in Japanese.

Please use your browser's translation (or DeepL/ChatGPT) to read it!

Hope it helps!

# データの読み込み・ライブラリのインポート

In [23]:
import pandas as pd
import numpy as np

import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


In [24]:
import kagglehub

path = kagglehub.competition_download("playground-series-s6e9")

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/playground-series-s6e9


In [25]:
train = pd.read_csv("../input/competitions/playground-series-s6e9/train.csv")
test = pd.read_csv("../input/competitions/playground-series-s6e9/test.csv")
sample_submission = pd.read_csv("../input/competitions/playground-series-s6e9/sample_submission.csv")

# 関数の定義

## ハイパーパラメータ

- `max_depth` : 3 が底。2 ～ 3 個の変数の組み合わせが限界、それ以上は過学習する。
- `num_leaves` : `max_depth` に対応。

In [26]:
def train_lgb(train_df, test_df, feature_cols, target_col="Will_Buy_EV"):
    """
    指定された特徴量で LightGBM の交差検証学習を行い、
    OOF予測値（確率）、テスト予測値（確率）、学習済みモデル群を返す関数
    """
    # 1. 目的変数の変換
    y_train = (train_df[target_col] == "Yes").astype(int).values # NumPy配列にして高速化
    
    # 【高速化①】ループの前にカテゴリ型への一括変換を済ませる
    X_train_df = train_df[feature_cols].copy()
    X_test_df = test_df[feature_cols].copy()
    
    for col in feature_cols:
        if X_train_df[col].dtype == 'object':
            X_train_df[col] = X_train_df[col].astype('category')
            X_test_df[col] = X_test_df[col].astype('category')

    # 【高速化②】.iloc のオーバーヘッドを避けるため、一回 pandas のままではなく 
    # LightGBM が最も得意とする DataFrame/NumPy の構造を維持する
    X_train_df = X_train_df.reset_index(drop=True)

    cv = KFold(n_splits=5, shuffle=True, random_state=0)
    
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'max_depth': 3,
        'num_leaves': 8,
        'learning_rate': 0.1,
        # 'max_bin': 31,

        # 🚀 過学習防止 & 高速化（ランダム性の導入）
        'colsample_bytree': 0.8,
        # 'subsample': 0.8,
        # 'subsample_freq': 1, # 💡 subsampleを動かす時は、1イテレーションごとにサンプリングし直すこの指定をセットで入れる
        
        'n_jobs': -1,  # 全コア並列使用
        'random_state': 0,
        'verbose': -1,
        'enable_categorical': True,

        # 🚀 GPUを使用するための設定
        # 'device_type': 'gpu',
        # 'gpu_use_dp': False,  # 必ず False（単精度/float32）にする。倍精度(True)は極端に遅くなります
    }

    models = []
    oof_train = np.zeros(len(X_train_df))

    # 💡 ループ内での y_preds への append は廃止
    
    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train_df, y_train)):
        # 【高速化③】iloc のスライスを最小限に
        X_tr = X_train_df.iloc[train_idx]
        y_tr = y_train[train_idx]
        X_va = X_train_df.iloc[valid_idx]
        y_va = y_train[valid_idx]

        # Datasetの作成 (free_raw_data=True でメモリを節約)
        lgb_train = lgb.Dataset(X_tr, y_tr, free_raw_data=True)
        lgb_eval = lgb.Dataset(X_va, y_va, reference=lgb_train, free_raw_data=True)

        # モデルの学習
        model = lgb.train(
            params,
            lgb_train,
            valid_sets=[lgb_eval], # lgb_train を削除して検証データのみにする
            num_boost_round=1500,  # デフォルトは 1000 あたり

            # stopping_rounds のデフォルトは 100 あたり
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        best_iter = model.best_iteration
        best_score = model.best_score['valid_0']['auc']
        print(f"Fold {fold+1} | Best Iteration: {best_iter:4d} | Best AUC: {best_score:.5f}")

        # 検証データの予測
        oof_train[valid_idx] = model.predict(X_va, num_iteration=model.best_iteration)
        
        # 💡　test_df の予測処理を削除し、モデルの保存だけに限定
        models.append(model)

    # 全体の OOF ROC-AUC スコアを計算
    score = roc_auc_score(y_train, oof_train)
    print(f"OOF ROC-AUC Score: {score:.5f}")

    # 💡 すべての学習が終わった後、ループの外でテストデータを一括予測する
    print("\nLoop finished. Predicting test data...")
    y_preds = [model.predict(X_test_df, num_iteration=model.best_iteration) for model in models]
    y_preds_mean = np.mean(y_preds, axis=0)

    return oof_train, y_preds_mean, models


# EDA

## binning

### commute_bin

通勤距離のビン分け

In [27]:
# 通勤距離のビン分け（範囲の定義）
# 0〜8km（5kmの大量データ含む）、8〜20km、20〜60km（中距離）、60〜80km（長距離）、80km〜（超長距離）
bins = [0, 8, 20, 60, 80, float('inf')]
labels = ['short_5km_zone', 'short_mid', 'mid_distance', 'long_distance', 'super_long_distance']

# pd.cut でカテゴリ変数化（LGBMで直接扱えるように category 型にキャスト）
train['commute_bin'] = pd.cut(train['Daily_Commute_km'], bins=bins, labels=labels).astype('category')
test['commute_bin'] = pd.cut(test['Daily_Commute_km'], bins=bins, labels=labels).astype('category')

### income_bin

年収のビン分け

In [28]:
income_bins = [0, 30000, 60000, 100000, 150000, float('inf')]
income_labels = ['income_min_30k', 'low_income', 'mid_income', 'high_income', 'super_high_income']

train['income_bin'] = pd.cut(train['Annual_Income_USD'], bins=income_bins, labels=income_labels).astype('category')
test['income_bin'] = pd.cut(test['Annual_Income_USD'], bins=income_bins, labels=income_labels).astype('category')

## env_x_anxiety_cat

環境意識 × 不安度（心理的バランス）

In [29]:
# Range_Anxiety_Level の数値化
anxiety_map = {'Low': 1, 'Medium': 2, 'High': 3}

train['range_anxiety_num'] = train['Range_Anxiety_Level'].map(anxiety_map)
test['range_anxiety_num'] = test['Range_Anxiety_Level'].map(anxiety_map)

In [30]:
# "5.0_Low", "1.0_High" のような 15 通りの組み合わせを作成
train['env_x_anxiety_cat'] = (train['Environmental_Concern_Level'].astype(str) + '_' + train['Range_Anxiety_Level'].astype(str)).astype('category')
test['env_x_anxiety_cat'] = (test['Environmental_Concern_Level'].astype(str) + '_' + test['Range_Anxiety_Level'].astype(str)).astype('category')

## Charging_Convenience

`Charging_Stations_Near_Home` + `Charging_Stations_Near_Work`

In [31]:
train["Charging_Convenience"] = train["Charging_Stations_Near_Home"] + train["Charging_Stations_Near_Work"]
test["Charging_Convenience"] = test["Charging_Stations_Near_Home"] + test["Charging_Stations_Near_Work"]

## income_per_car

1 台あたりの資金的余裕

In [32]:
# 0 除算を防ぐため np.where を使用（0台の場合はそのまま年収、または np.nan / 特定値に）
train['income_per_car'] = train['Annual_Income_USD'] / np.where(
    train['Number_of_Cars_Owned'] == 0, 1, train['Number_of_Cars_Owned']
)
test['income_per_car'] = test['Annual_Income_USD'] / np.where(
    test['Number_of_Cars_Owned'] == 0, 1, test['Number_of_Cars_Owned']
)

## gender_x_city

都市型 × 性別の生活スタイル

In [33]:
train['gender_x_city'] = (
    train['Gender'].astype(str) + '_' + train['City_Type'].astype(str)
).astype('category')
test['gender_x_city'] = (
    test['Gender'].astype(str) + '_' + test['City_Type'].astype(str)
).astype('category')

## gender_x_car

現在の車種 × 性別の生活スタイル

In [34]:
train['gender_x_car'] = (
    train['Gender'].astype(str) + '_' + train['Current_Car_Type'].astype(str)
).astype('category')
test['gender_x_car'] = (
    test['Gender'].astype(str) + '_' + test['Current_Car_Type'].astype(str)
).astype('category')

## Buy_score

[https://www.kaggle.com/competitions/playground-series-s6e9/discussion/739303](http://)

In [35]:
train['Buy_score'] = (
    1.2 * (train['Annual_Income_USD'] / 100000)
    + 0.6 * train['Environmental_Concern_Level'] 
    + 2 * (train['Subsidy_Available'] == 'Yes').astype(int)
    - 1 * (train['Range_Anxiety_Level'] == 'Medium').astype(int)
    - 3 * (train['Range_Anxiety_Level'] == 'High').astype(int)
)

test['Buy_score'] = (
    1.2 * (test['Annual_Income_USD'] / 100000)
    + 0.6 * test['Environmental_Concern_Level'] 
    + 2 * (test['Subsidy_Available'] == 'Yes').astype(int)
    - 1 * (test['Range_Anxiety_Level'] == 'Medium').astype(int)
    - 3 * (test['Range_Anxiety_Level'] == 'High').astype(int)
)

### Score_Economic

In [36]:
train['Score_Economic'] = (
    1.2 * (train['Annual_Income_USD'] / 100000)
    + 2 * (train['Subsidy_Available'] == 'Yes').astype(int)
)

test['Score_Economic'] = (
    1.2 * (test['Annual_Income_USD'] / 100000)
    + 2 * (test['Subsidy_Available'] == 'Yes').astype(int)
)

### Score_Psychological

In [37]:
train['Score_Psychological'] = (
    0.6 * train['Environmental_Concern_Level']
    - 1 * (train['Range_Anxiety_Level'] == 'Medium').astype(int)
    - 3 * (train['Range_Anxiety_Level'] == 'High').astype(int)
)

test['Score_Psychological'] = (
    0.6 * test['Environmental_Concern_Level']
    - 1 * (test['Range_Anxiety_Level'] == 'Medium').astype(int)
    - 3 * (test['Range_Anxiety_Level'] == 'High').astype(int)
)

# スコアリング特徴量

全体購入率 17.5% を基準に閾値を決める。
25% 以上（全体購入率の 1.4 倍以上）から、購入率が高いと設定。

In [38]:
# 1. プラス要素のカウント（購入率 25% 以上の条件）
train['Pos_score'] = 0
test['Pos_score'] = 0

# 重み調整用の変数
num1 = 1
num2 = 1.5
num3 = 2

# ゆるいプラス条件（購入率 25 ~ 35%） -> +1 点
train['Pos_score'] += (train['Subsidy_Available'] == 'Yes').astype(int) * num1
test['Pos_score'] += (test['Subsidy_Available'] == 'Yes').astype(int) * num1
train['Pos_score'] += (train['Environmental_Concern_Level'] == 4.0).astype(int) * num1
test['Pos_score'] += (test['Environmental_Concern_Level'] == 4.0).astype(int) * num1
train['Pos_score'] += (train['commute_bin'] == 'short_mid').astype(int) * num1
test['Pos_score'] += (test['commute_bin'] == 'short_mid').astype(int) * num1
train['Pos_score'] += (train['income_bin'] == 'high_income').astype(int) * num1
test['Pos_score'] += (test['income_bin'] == 'high_income').astype(int) * num1
train['Pos_score'] += (train['env_x_anxiety_cat'] == '4.0_Low').astype(int) * num1
test['Pos_score'] += (test['env_x_anxiety_cat'] == '4.0_Low').astype(int) * num1

# 強力なプラス条件（購入率 35 ~ 50%） -> +2 点
train['Pos_score'] += (train['income_bin'] == 'super_high_income').astype(int) * num2
test['Pos_score'] += (test['income_bin'] == 'super_high_income').astype(int) * num2

# 非常に強力なプラス条件（購入率 50% 以上） -> +3 点
train['Pos_score'] += (train['Environmental_Concern_Level'] == 5.0).astype(int) * num3
test['Pos_score'] += (test['Environmental_Concern_Level'] == 5.0).astype(int) * num3
train['Pos_score'] += (train['env_x_anxiety_cat'] == '5.0_Low').astype(int) * num3
test['Pos_score'] += (test['env_x_anxiety_cat'] == '5.0_Low').astype(int) * num3

In [39]:
# 2. マイナス要素のカウント（購入率が極端に低い 10% 未満の条件）
train['Neg_score'] = 0
test['Neg_score'] = 0


# ゆるいマイナス条件（購入率 7.5 ~ 10% 未満）
train['Neg_score'] += (train['income_bin'] == 'low_income').astype(int) * num1
test['Neg_score'] += (test['income_bin'] == 'low_income').astype(int) * num1


# 強力なマイナス条件（購入率 2.5 ~ 7.5% 未満）
train['Neg_score'] += (train['range_anxiety_num'] >= 2.0).astype(int) * num2
test['Neg_score'] += (test['range_anxiety_num'] >= 2.0).astype(int) * num2
train['Neg_score'] += (train['income_bin'] == 'income_min_30k').astype(int) * num2
test['Neg_score'] += (test['income_bin'] == 'income_min_30k').astype(int) * num2


# 非常に強力なマイナス条件（購入率 2.5% 未満）
train['Neg_score'] += (train['Subsidy_Available'] == 'No').astype(int) * num3
test['Neg_score'] += (test['Subsidy_Available'] == 'No').astype(int) * num3
train['Neg_score'] += (train['Environmental_Concern_Level'] <= 2.0).astype(int) * num3
test['Neg_score'] += (test['Environmental_Concern_Level'] <= 2.0).astype(int) * num3

In [40]:
# 3. プラス要素とマイナス要素の差分
# 学習に使うのはこれだけ、Pos_score と Neg_score はバリアンスが高いので中間変数としてのみ活用
train['Total_score'] = train['Pos_score'] - train['Neg_score']
test['Total_score'] = test['Pos_score'] - test['Neg_score']

# クラスタリング

In [41]:
binary_map = {'Yes': 1, 'No': 0}

# train の変換
train['Home_Charging_Possible_num'] = (
    train['Home_Charging_Possible'].map(binary_map).astype(int)
)
train['Subsidy_Available_num'] = (
    train['Subsidy_Available'].map(binary_map).astype(int)
)

# test の変換
test['Home_Charging_Possible_num'] = (
    test['Home_Charging_Possible'].map(binary_map).astype(int)
)
test['Subsidy_Available_num'] = (
    test['Subsidy_Available'].map(binary_map).astype(int)
)

In [42]:
# 「都市 & 充電インフラ」の環境特徴量を定義
cluster_features = [
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Charging_Convenience",
    "Home_Charging_Possible_num",
]

# スケーリング（StandardScaler で標準化: KMeans に必須）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train[cluster_features])
X_test_scaled = scaler.transform(test[cluster_features])

# 指定したクラスタ数でクラスタリング実行
k = 4
kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)

train["infrastructure_cluster"] = kmeans.fit_predict(X_train_scaled)
test["infrastructure_cluster"] = kmeans.predict(X_test_scaled)

# LightGBM で扱えるように category 型に変換
train["infrastructure_cluster"] = train["infrastructure_cluster"].astype(
    "category"
)

test["infrastructure_cluster"] = test["infrastructure_cluster"].astype(
    "category"
)

# 実行

In [43]:
%%time

# 1. 不要な列の除外（特徴量の選択）
features = [
    col
    for col in train.columns
    if col not in [
    "id",
    "Will_Buy_EV",
    "Range_Anxiety_Level",
    'range_anxiety_num',
    'Home_Charging_Possible_num',
    'Subsidy_Available_num',
    'City_Type',
    'Current_Car_Type',
    'Pos_score',
    'Neg_score',
    ]
]

# 2. 【型変換の一括適用】Daily_Commute_kmの100倍整数化と、各列の型軽量化
train_copy = train.copy()
test_copy = test.copy()

# ① Daily_Commute_km を100倍して丸める
if 'Daily_Commute_km' in features:
    train_copy['Daily_Commute_km'] = (train_copy['Daily_Commute_km'] * 100).round()
    test_copy['Daily_Commute_km'] = (test_copy['Daily_Commute_km'] * 100).round()

# ② 辞書型を用いて一括型キャスト
type_mapping = {}
for col in features:
    if train_copy[col].dtype == 'object':
        type_mapping[col] = 'category'
    elif col in ['Annual_Income_USD', 'Daily_Commute_km']:
        type_mapping[col] = 'int32'
    elif col == 'Environmental_Concern_Level':
        type_mapping[col] = 'int16'

train_copy = train_copy.astype(type_mapping)
test_copy = test_copy.astype(type_mapping)


# 3. 余計な列を完全に排除した「純粋な軽量DataFrame」を孤立させて作成
pure_train = train_copy[features + ["Will_Buy_EV"]].copy()
pure_test = test_copy[features].copy()


# 4. 学習の実行（軽量化した pure_train / pure_test を投入）
oof_train, y_preds, models = train_lgb(pure_train, pure_test, features)


# 5. 正解ラベルの数値化と全体の OOF ROC-AUC スコアの計算
y_true = (pure_train["Will_Buy_EV"] == "Yes").astype(int)
cv_auc = roc_auc_score(y_true, oof_train)
print(f"🔥 Final CV ROC-AUC Score: {cv_auc:.5f}\n")


# 6. 特徴量重要度の可視化（表形式で出力）
feature_importances = np.mean(
    [model.feature_importance(importance_type="gain") for model in models],
    axis=0,
)

importance_df = (
    pd.DataFrame({"feature": features, "importance": feature_importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(20))

Fold 1 | Best Iteration: 1114 | Best AUC: 0.94161
Fold 2 | Best Iteration: 1174 | Best AUC: 0.94173
Fold 3 | Best Iteration: 1108 | Best AUC: 0.94239
Fold 4 | Best Iteration:  966 | Best AUC: 0.94159
Fold 5 | Best Iteration: 1131 | Best AUC: 0.94397
OOF ROC-AUC Score: 0.94224

Loop finished. Predicting test data...
🔥 Final CV ROC-AUC Score: 0.94224



,feature,importance
0,Buy_score,945544.011818
1,Total_score,327812.617989
2,Score_Economic,28431.780707
3,Daily_Commute_km,20412.895066
4,Annual_Income_USD,18649.192225
5,Subsidy_Available,6863.778296
6,Age,6283.017633
7,env_x_anxiety_cat,4035.020669
8,income_per_car,3495.266978
9,gender_x_car,1863.576239


CPU times: user 17min 29s, sys: 1.83 s, total: 17min 31s
Wall time: 4min 30s


# 提出

In [44]:
sub = sample_submission.copy()
sub["Will_Buy_EV"] = y_preds

# CSVファイルとして保存
sub.to_csv("submission.csv", index=False)
print("\nsubmission.csv を保存しました！")


submission.csv を保存しました！
